# LAB 07 — From Attention to Transformer
**Môn học:** Xử lý ngôn ngữ tự nhiên và ứng dụng

Mục tiêu: xây dựng Transformer Encoder tối giản cho phân loại cảm xúc IMDB; khảo sát Positional Encoding, số lượng Attention Heads, độ dài chuỗi, pooling và so sánh với LSTM.

Notebook kế thừa quy trình IMDB/tokenization từ LAB05 và ý tưởng Q-K-V/Scaled Dot-Product Attention từ LAB06. **Không dùng `nn.TransformerEncoderLayer` cho phần lõi.**

> Chạy tuần tự từ trên xuống. Nên chọn GPU trên Colab. Các phần TODO về tính toán, dự đoán và diễn giải cần tự hoàn thành theo yêu cầu học phần.

## 0. Cài đặt và import

In [ ]:
# Nếu chạy trên Colab và chưa có thư viện:
# !pip -q install datasets scikit-learn matplotlib pandas

import math, time, random
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 1. Tải và tiền xử lý IMDB
Tạo validation từ tập train; giữ nguyên test set để đánh giá cuối cùng. Tokenizer dưới đây là baseline đơn giản theo LAB05.

In [ ]:
raw = load_dataset("imdb")
split = raw["train"].train_test_split(test_size=0.1, seed=SEED)
train_raw, valid_raw, test_raw = split["train"], split["test"], raw["test"]

def tokenize(text):
    return text.lower().replace("<br />", " ").split()

MAX_VOCAB = 30000
PAD, UNK = "<pad>", "<unk>"
counts = Counter()
for row in train_raw:
    counts.update(tokenize(row["text"]))
vocab = {PAD: 0, UNK: 1}
for word, _ in counts.most_common(MAX_VOCAB - 2):
    vocab[word] = len(vocab)

def encode(text, max_len):
    ids = [vocab.get(w, vocab[UNK]) for w in tokenize(text)[:max_len]]
    return torch.tensor(ids or [vocab[UNK]], dtype=torch.long)

class IMDBDataset(Dataset):
    def __init__(self, rows, max_len=128):
        self.rows, self.max_len = rows, max_len
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        row = self.rows[i]
        return encode(row["text"], self.max_len), int(row["label"]), row["text"]

def collate_fn(batch):
    xs, ys, texts = zip(*batch)
    lengths = torch.tensor([len(x) for x in xs])
    xs = pad_sequence(xs, batch_first=True, padding_value=vocab[PAD])
    return xs, torch.tensor(ys), lengths, list(texts)

BATCH_SIZE = 64
MAX_LEN = 128
def make_loaders(max_len=MAX_LEN):
    train_ds = IMDBDataset(train_raw, max_len)
    valid_ds = IMDBDataset(valid_raw, max_len)
    test_ds = IMDBDataset(test_raw, max_len)
    return (
        DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn),
        DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn),
        DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    )
train_loader, valid_loader, test_loader = make_loaders()
print("Vocab:", len(vocab), "| sizes:", len(train_raw), len(valid_raw), len(test_raw))

## 2. Positional Encoding
Cộng vector vị trí vào embedding để đưa thông tin thứ tự token vào biểu diễn.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32)
                        * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div[:pe[:, 1::2].shape[1]])
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

# Vẽ positional encoding
pe = PositionalEncoding(128, 128, 0).pe[0].numpy()
plt.figure(figsize=(10,4))
plt.imshow(pe.T, aspect="auto", origin="lower")
plt.xlabel("Position"); plt.ylabel("Embedding dimension")
plt.title("Sinusoidal Positional Encoding"); plt.colorbar()
plt.tight_layout(); plt.savefig("positional_encoding.png", dpi=160); plt.show()

## 3. Multi-Head Self-Attention
Với mỗi head: Q=XWQ, K=XWK, V=XWV; Attention=softmax(QKᵀ/√dk); output=Attention·V. Sau đó nối các head và chiếu qua WO.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model=128, n_heads=4, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model, self.n_heads = d_model, n_heads
        self.d_k = d_model // n_heads
        self.WQ = nn.Linear(d_model, d_model)
        self.WK = nn.Linear(d_model, d_model)
        self.WV = nn.Linear(d_model, d_model)
        self.WO = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)
        self.last_attention = None
    def forward(self, x, padding_mask=None):
        B, T, _ = x.shape
        def split(layer):
            return layer(x).view(B,T,self.n_heads,self.d_k).transpose(1,2)
        Q, K, V = split(self.WQ), split(self.WK), split(self.WV)
        scores = Q @ K.transpose(-2,-1) / math.sqrt(self.d_k)
        if padding_mask is not None:
            scores = scores.masked_fill(padding_mask[:,None,None,:], -1e9)
        weights = torch.softmax(scores, dim=-1)
        self.last_attention = weights.detach()
        context = self.drop(weights) @ V
        context = context.transpose(1,2).contiguous().view(B,T,self.d_model)
        return self.WO(context)

## 4. Transformer Encoder Block và Classifier
Block gồm Attention → Residual + LayerNorm → FFN → Residual + LayerNorm. Mean pooling bỏ qua token padding.

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model=128, n_heads=4, d_ff=256, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.drop1, self.drop2 = nn.Dropout(dropout), nn.Dropout(dropout)
        self.ffn = nn.Sequential(nn.Linear(d_model,d_ff), nn.ReLU(),
                                 nn.Linear(d_ff,d_model))
    def forward(self, x, padding_mask=None):
        x = self.norm1(x + self.drop1(self.attn(x, padding_mask)))
        x = self.norm2(x + self.drop2(self.ffn(x)))
        return x

class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4, d_ff=256,
                 n_layers=2, max_len=256, dropout=0.1, pooling="mean"):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos = PositionalEncoding(d_model, max_len, dropout)
        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(d_model,n_heads,d_ff,dropout)
            for _ in range(n_layers)])
        self.fc = nn.Linear(d_model,2)
        self.pooling = pooling
    def forward(self, ids, lengths=None):
        mask = ids.eq(0)
        x = self.pos(self.embedding(ids))
        for block in self.blocks:
            x = block(x, mask)
        if self.pooling == "mean":
            keep = (~mask).unsqueeze(-1)
            x = (x * keep).sum(1) / keep.sum(1).clamp(min=1)
        else:
            x = x[:,0,:]  # first-token pooling, không phải pretrained CLS token
        return self.fc(x)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

model = TransformerClassifier(len(vocab)).to(DEVICE)
print("Output shape:", model(torch.randint(1,len(vocab),(8,100)).to(DEVICE)).shape)
print("Trainable parameters:", count_params(model))

## 5. Hàm huấn luyện và đánh giá

In [ ]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    criterion = nn.CrossEntropyLoss()
    losses, ys, preds, texts = [], [], [], []
    start = time.time()
    for x,y,lengths,batch_texts in loader:
        x,y = x.to(DEVICE), y.to(DEVICE)
        if training: optimizer.zero_grad()
        with torch.set_grad_enabled(training):
            logits = model(x,lengths)
            loss = criterion(logits,y)
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(),1.0)
                optimizer.step()
        losses.append(loss.item())
        ys.extend(y.cpu().tolist()); preds.extend(logits.argmax(1).cpu().tolist())
        texts.extend(batch_texts)
    p,r,f1,_ = precision_recall_fscore_support(ys,preds,average="binary",zero_division=0)
    return {"loss":float(np.mean(losses)), "accuracy":accuracy_score(ys,preds),
            "precision":p, "recall":r, "f1":f1, "seconds":time.time()-start,
            "y":ys, "pred":preds, "texts":texts}

def fit(model, train_loader, valid_loader, epochs=3, lr=3e-4):
    opt = torch.optim.AdamW(model.parameters(),lr=lr)
    history=[]
    for epoch in range(1,epochs+1):
        tr=run_epoch(model,train_loader,opt)
        va=run_epoch(model,valid_loader)
        history.append({"epoch":epoch,"train_loss":tr["loss"],"valid_loss":va["loss"],
                        "valid_accuracy":va["accuracy"],"valid_f1":va["f1"]})
        print(f"Epoch {epoch}: train_loss={tr['loss']:.4f}, valid_loss={va['loss']:.4f}, "
              f"val_acc={va['accuracy']:.4f}, val_f1={va['f1']:.4f}")
    return pd.DataFrame(history)

## 6. Huấn luyện Transformer chính

In [ ]:
transformer = TransformerClassifier(len(vocab), d_model=128, n_heads=4,
                                       n_layers=2, max_len=MAX_LEN).to(DEVICE)
history = fit(transformer, train_loader, valid_loader, epochs=3)
history.to_csv("learning_history.csv",index=False)
plt.figure(figsize=(7,4))
plt.plot(history.epoch,history.train_loss,marker="o",label="Train")
plt.plot(history.epoch,history.valid_loss,marker="o",label="Validation")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Learning Curves")
plt.legend(); plt.tight_layout(); plt.savefig("learning_curves.png",dpi=160); plt.show()

test_metrics = run_epoch(transformer,test_loader)
print({k:round(test_metrics[k],4) for k in ["accuracy","precision","recall","f1","seconds"]})
cm=confusion_matrix(test_metrics["y"],test_metrics["pred"])
ConfusionMatrixDisplay(cm,display_labels=["negative","positive"]).plot()
plt.title("Transformer — IMDB Test Confusion Matrix")
plt.tight_layout(); plt.savefig("confusion_matrix.png",dpi=160); plt.show()

## 7. Experiment — Số lượng Attention Heads
Giữ d_model=128, thử 1, 2, 4, 8 heads. Mặc định mỗi cấu hình chạy một epoch để khảo sát nhanh; cần tăng epochs nếu muốn so sánh kết quả ổn định hơn.

In [ ]:
head_results=[]
for heads in [1,2,4,8]:
    m=TransformerClassifier(len(vocab),d_model=128,n_heads=heads,max_len=MAX_LEN).to(DEVICE)
    opt=torch.optim.AdamW(m.parameters(),lr=3e-4)
    tr=run_epoch(m,train_loader,opt); va=run_epoch(m,valid_loader)
    head_results.append({"experiment":"heads","value":heads,"parameters":count_params(m),
                         "train_seconds":tr["seconds"],"accuracy":va["accuracy"],"f1":va["f1"]})
heads_df=pd.DataFrame(head_results)
display(heads_df)

## 8. Experiment — Trực quan hóa các Attention Heads
Chọn một mẫu trong test batch. Quan sát ma trận từng head; không tự gán chức năng ngôn ngữ nếu hình không chứng minh điều đó.

In [ ]:
transformer.eval()
x,y,lengths,texts=next(iter(test_loader))
with torch.no_grad(): _=transformer(x[:1].to(DEVICE))
attn=transformer.blocks[0].attn.last_attention[0].cpu().numpy()
fig,axes=plt.subplots(1,attn.shape[0],figsize=(4*attn.shape[0],3))
if attn.shape[0]==1: axes=[axes]
for h,ax in enumerate(axes):
    ax.imshow(attn[h],aspect="auto",cmap="viridis")
    ax.set_title(f"Head {h+1}"); ax.set_xlabel("Key position"); ax.set_ylabel("Query position")
plt.tight_layout(); plt.savefig("attention_heads.png",dpi=160); plt.show()
print(texts[0][:500])

## 9. Experiment — So sánh LSTM và Transformer
Dùng chung dữ liệu, tokenizer, split, batch size và số epoch. Nếu LAB05 có LSTM baseline cụ thể, thay class bên dưới bằng đúng kiến trúc LAB05 để đảm bảo so sánh.

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self,vocab_size,emb_dim=128,hidden=128):
        super().__init__()
        self.emb=nn.Embedding(vocab_size,emb_dim,padding_idx=0)
        self.lstm=nn.LSTM(emb_dim,hidden,batch_first=True,bidirectional=True)
        self.drop=nn.Dropout(0.2); self.fc=nn.Linear(hidden*2,2)
    def forward(self,ids,lengths=None):
        out,_=self.lstm(self.emb(ids))
        mask=ids.ne(0).unsqueeze(-1)
        pooled=(out*mask).sum(1)/mask.sum(1).clamp(min=1)
        return self.fc(self.drop(pooled))

lstm=LSTMClassifier(len(vocab)).to(DEVICE)
lstm_history=fit(lstm,train_loader,valid_loader,epochs=3)
lstm_test=run_epoch(lstm,test_loader)
comparison=pd.DataFrame([
 {"model":"LSTM","parameters":count_params(lstm),**{k:lstm_test[k] for k in ["accuracy","precision","recall","f1","seconds"]}},
 {"model":"Transformer","parameters":count_params(transformer),**{k:test_metrics[k] for k in ["accuracy","precision","recall","f1","seconds"]}}
])
display(comparison)

## 10. Experiment — Độ dài chuỗi 32, 64, 128, 256
Attention matrix có T×T phần tử trên mỗi head và mỗi mẫu; độ phức tạp theo chiều dài chuỗi là O(T²). Mỗi cấu hình dưới đây huấn luyện nhanh một epoch.

In [ ]:
length_results=[]
for seq_len in [32,64,128,256]:
    tr_loader,va_loader,_=make_loaders(seq_len)
    m=TransformerClassifier(len(vocab),d_model=128,n_heads=4,max_len=seq_len).to(DEVICE)
    opt=torch.optim.AdamW(m.parameters(),lr=3e-4)
    tr=run_epoch(m,tr_loader,opt); va=run_epoch(m,va_loader)
    length_results.append({"experiment":"sequence_length","value":seq_len,
        "attention_elements_per_head":seq_len**2,"parameters":count_params(m),
        "train_seconds":tr["seconds"],"accuracy":va["accuracy"],"f1":va["f1"]})
length_df=pd.DataFrame(length_results)
display(length_df)

## 11. Experiment — Pooling
So sánh mean pooling và first-token pooling (CLS-style approximation). Đây không phải pretrained [CLS] representation; token đầu tiên là token văn bản thông thường trong setup này.

In [ ]:
pool_results=[]
for pooling in ["mean","first"]:
    m=TransformerClassifier(len(vocab),n_heads=4,max_len=MAX_LEN,pooling=pooling).to(DEVICE)
    opt=torch.optim.AdamW(m.parameters(),lr=3e-4)
    tr=run_epoch(m,train_loader,opt); va=run_epoch(m,valid_loader)
    pool_results.append({"experiment":"pooling","value":pooling,"parameters":count_params(m),
                         "train_seconds":tr["seconds"],"accuracy":va["accuracy"],"f1":va["f1"]})
pool_df=pd.DataFrame(pool_results)
display(pool_df)

## 12. Lưu kết quả và trích xuất lỗi
Các con số chỉ xuất hiện sau khi chạy thật. Lưu CSV và lấy ba ví dụ đúng/sai để phân tích.

In [ ]:
results=[]
results.append({"experiment":"main_transformer","value":4,"parameters":count_params(transformer),
                "train_seconds":test_metrics["seconds"],"accuracy":test_metrics["accuracy"],
                "precision":test_metrics["precision"],"recall":test_metrics["recall"],"f1":test_metrics["f1"]})
results += head_results + length_results + pool_results
results.append({"experiment":"lstm_baseline","value":"LSTM","parameters":count_params(lstm),
                "train_seconds":lstm_test["seconds"],"accuracy":lstm_test["accuracy"],
                "precision":lstm_test["precision"],"recall":lstm_test["recall"],"f1":lstm_test["f1"]})
pd.DataFrame(results).to_csv("results.csv",index=False)

pred_df=pd.DataFrame({"text":test_metrics["texts"],"actual":test_metrics["y"],
                      "predicted":test_metrics["pred"]})
pred_df["correct"]=pred_df.actual==pred_df.predicted
print("3 examples predicted correctly:")
display(pred_df[pred_df.correct].head(3))
print("3 examples predicted incorrectly:")
display(pred_df[~pred_df.correct].head(3))

## 13. TODO — Phân tích theo yêu cầu LAB07

1. Hoàn thiện phép tính Q, K, V, scores, softmax và output trong `calculations.md` theo số liệu đề bài.
2. Điền sáu dự đoán trước thực nghiệm vào `prediction.md` (Prediction, Reason, Confidence). Không sửa dự đoán ban đầu sau khi xem kết quả.
3. Phân tích hình Positional Encoding; so sánh mô hình có/không có positional encoding và phép hoán vị token nếu triển khai thêm ablation.
4. So sánh các attention heads: head nào tập trung/phân tán/tương tự/khác biệt? Dẫn chứng từ hình.
5. So sánh LSTM và Transformer theo số tham số, thời gian, Accuracy, Precision, Recall, F1. Không kết luận Transformer luôn tốt hơn.
6. Phân tích ảnh hưởng sequence length 32/64/128/256 và liên hệ T².
7. Chọn ít nhất 3 dự đoán đúng và 3 dự đoán sai; phân loại lỗi: phủ định, tương phản, câu dài, truncation, từ hiếm, cảm xúc mơ hồ, thuật ngữ miền hoặc nhiễu.
8. So sánh mean pooling và first-token pooling.
9. Hoàn thiện reflection: Q/K/V, Multi-Head Attention, Positional Encoding, Residual, LayerNorm, FFN và giới hạn của mô hình.

**Lưu ý:** Attention weights là phân phối trọng số, không tự nó chứng minh lời giải thích nhân quả cho dự đoán của mô hình.